In [1]:
import random
import json
import re
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix
)

# -----------------------------
# Simulated AI Model
# -----------------------------
def call_ai_api(payload, error_rate=0.0):
    """
    Simulated reasoning model.
    Detects category of failure from log text.
    error_rate: probability of returning a wrong prediction (to simulate noise).
    """

    log = payload["log"]
    msg = log.lower()

    # Intentional noise (simulate imperfect real-world model)
    if random.random() < error_rate:
        return {"cause": random.choice(["system_error", "network_failure", "db_error", "auth_issue", "normal_op"])}

    # Pattern-based classification simulation
    if "disk" in msg or "io" in msg or "kernel panic" in msg:
        return {"cause": "system_error"}

    if "timeout" in msg or "connection refused" in msg:
        return {"cause": "network_failure"}

    if "db" in msg or "query failed" in msg:
        return {"cause": "db_error"}

    if "unauthorized" in msg or "forbidden" in msg or "login failed" in msg:
        return {"cause": "auth_issue"}

    return {"cause": "normal_op"}


# -----------------------------
# Dataset Generator
# -----------------------------
ERROR_TYPES = {
    "system_error": [
        "Disk Failure", "Kernel Panic", "I/O error", "Memory corruption"
    ],
    "network_failure": [
        "Timeout", "Connection refused", "Packet loss detected"
    ],
    "db_error": [
        "DB connection lost", "Query failed", "Transaction rollback"
    ],
    "auth_issue": [
        "Unauthorized access", "Login failed", "Forbidden request"
    ],
    "normal_op": [
        "OK", "Job completed", "Heartbeat signal", "Routine check"
    ]
}

def make_dataset(n=300):
    data = []
    labels = []

    all_classes = list(ERROR_TYPES.keys())

    for _ in range(n):
        label = random.choice(all_classes)
        message = random.choice(ERROR_TYPES[label])

        log = json.dumps({
            "level": "ERROR" if label != "normal_op" else "INFO",
            "message": message
        })

        data.append(log)
        labels.append(label)

    return data, labels


# -----------------------------
# Run Inference
# -----------------------------
random.seed(42)
np.random.seed(42)

logs, labels = make_dataset()

preds = []
for log in logs:
    out = call_ai_api({"log": log}, error_rate=0.05)   # small noise
    preds.append(out["cause"])

# -----------------------------
# Evaluation
# -----------------------------
accuracy = accuracy_score(labels, preds)
prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
cm = confusion_matrix(labels, preds)

print("Reasoner Accuracy: ", accuracy)
print("Precision:", prec)
print("Recall:", rec)
print("F1 Score:", f1)
print("\nConfusion Matrix:\n", cm)


Reasoner Accuracy:  0.68
Precision: 0.7708013199369582
Recall: 0.68
F1 Score: 0.6501367102952009

Confusion Matrix:
 [[55  0  1  0  0]
 [ 1 20  1  1 37]
 [ 0  0 14 21 21]
 [ 0  0  2 62  1]
 [ 0  0  0 10 53]]
